In [11]:
import json
import re
from typing import List
from tqdm import tqdm
from pydantic import BaseModel, Field
from langchain import PromptTemplate, LLMChain
from langchain.llms.ollama import Ollama
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import OutputParserException
import os

# Load jobs from JSON
with open("./data/shuffled_data.json", "r", encoding="utf-8") as file:
    jobs_list = json.load(file)

# Assign ID to each job if not already there
for index, job in enumerate(jobs_list, start=1):
    job["id"] = job.get("id", index)

# Define schemas
class Skill(BaseModel):
    skill: str
    influence: int = Field(..., ge=0, le=100)

class JobSkills(BaseModel):
    soft_skills: List[Skill]
    hard_skills: List[Skill]

output_parser = PydanticOutputParser(pydantic_object=JobSkills)
format_instructions = output_parser.get_format_instructions().replace("{", "{{").replace("}", "}}")


# Prompt template
prompt_template = f"""
You are given the job description text from a job posting. Your task is to extract the soft skills and hard skills required for the job mentioned in the job description.
Soft skills include communication, teamwork, adaptability, problem-solving, leadership, emotional intelligence, and time management.
Hard skills include (but are not limited to) the following:
- Programming and Software Development
- Data Analysis and Statistical Analysis
- Machine Learning and Artificial Intelligence
- SQL, Python, Java
- Cloud Computing (AWS, Azure, Google Cloud)
- Agile and Scrum Methodologies
- Programming and Software Development
- Data Analysis and Statistical Analysis
- Project Management
- Financial Analysis and Forecasting
- Technical Writing and Documentation
- Machine Learning and Artificial Intelligence
- Graphic Design and Visual Communication
- Digital Marketing and SEO/SEM
- Web Development
- Database Management and SQL
- Cybersecurity and Information Security
- IT Networking and Infrastructure Management
- Quality Assurance and Software Testing
- Computer-Aided Design (CAD) and 3D Modeling
- Engineering Design and Simulation
- Scientific Research and Laboratory Skills
- Legal Research and Compliance
- Social Media Management and Analytics
- Content Creation and Copywriting
- Multimedia Production and Video Editing
- Technical Support and Troubleshooting
- Operating Systems Administration
- DevOps and Continuous Integration/Deployment
- Agile and Scrum Methodologies
- Data Visualization
- Business Intelligence and Analytics
- Supply Chain Management and Logistics
- Sales and Negotiation Techniques
- Advanced Excel and Data Modeling
- Statistical Software Proficiency (R, SAS, SPSS)
- Cloud Computing (AWS, Azure, Google Cloud)
- Mobile Application Development
- Robotics and Automation Engineering
- Virtual Reality (VR) and Augmented Reality (AR) Development
- E-commerce Platform Management
- Digital Forensics and Incident Response
- Network Security Monitoring and Penetration Testing
- Biotechnology Techniques and Laboratory Procedures
- Geographic Information Systems (GIS) and Spatial Analysis
- Foreign Language Proficiency
- Medical Diagnosis and Patient Care
- Mechanical Engineering Design and Analysis
- Electronics Engineering and Circuit Design
- Management Consulting and Strategic Advisory

For each skill you extract, assign a percentage influence (0-100) based on how prominently it appears in the description.

Return ONLY the output in valid JSON format following this schema:
{format_instructions}

Job Description:
{{job_text}}
"""

template = PromptTemplate(input_variables=["job_text"], template=prompt_template)
llm = Ollama(model="llama3:8b", temperature=0, base_url="http://localhost:11434")
chain = LLMChain(llm=llm, prompt=template, output_parser=output_parser)

def clean_json_output(text: str) -> str:
    match = re.search(r'(\{.*\})', text, re.DOTALL)
    return match.group(1) if match else ""

# Ensure the export directory exists
os.makedirs('./data/export/', exist_ok=True)

def extract_skills(jobs_list: list, checkpoint_interval: int = 100) -> list:
    valid_jobs = []
    successful_parses = 0  # Counter for successfully parsed jobs
    for index, job in enumerate(tqdm(jobs_list, desc="Extracting skills"), start=1):
        
        # Always check if we should save the checkpoint, even if an error occurred
        if successful_parses % checkpoint_interval == 0 or index == len(jobs_list):
            if valid_jobs:  # Only save if there are valid jobs
                checkpoint_file = f"./data/export/jobs_with_skills_new_checkpoint_{index}.json"
                with open(checkpoint_file, "w", encoding="utf-8") as f:
                    json.dump(valid_jobs, f, indent=2)
                print(f"Checkpoint saved: {checkpoint_file}")
                
        job_desc = job.get("job_overview", "")
        if not job_desc:
            continue  # Skip jobs with no description

        try:
            result = chain.run(job_text=job_desc)
            skills = result.model_dump()

            # Check if the job has valid soft_skills and hard_skills
            if skills.get("soft_skills") and skills.get("hard_skills"):
                job["soft_skills"] = skills.get("soft_skills")
                job["hard_skills"] = skills.get("hard_skills")
                valid_jobs.append(job)
                successful_parses += 1  # Increment the successful parse counter
                print(f"Currently jobs added: {successful_parses}")
        except OutputParserException as e:
            raw_output = getattr(e, "llm_output", "")
            try:
                cleaned_output = clean_json_output(raw_output)
                skills = output_parser.parse(cleaned_output).model_dump()

                if skills.get("soft_skills") and skills.get("hard_skills"):
                    job["soft_skills"] = skills.get("soft_skills")
                    job["hard_skills"] = skills.get("hard_skills")
                    valid_jobs.append(job)
                    successful_parses += 1  # Increment the successful parse counter
                    print(f"Currently jobs added: {successful_parses}")
            except Exception as inner_exception:
                print(f"Error parsing job {job['id']} (skipped): {inner_exception}")
                continue  # Skip this job and proceed to the next one

    print(f"Total jobs successfully parsed: {successful_parses}")  # Output total count
    return valid_jobs


# Process the job list
valid_jobs_list = extract_skills(jobs_list)

# Final output (optional save)
with open("./data/export/jobs_with_skills_final.json", "w", encoding="utf-8") as f:
    json.dump(valid_jobs_list, f, indent=2)

print(f"Total valid jobs with skills: {len(valid_jobs_list)}")


Extracting skills:   0%|                                                           | 1/9646 [00:10<26:57:41, 10.06s/it]

Currently jobs added: 1


Extracting skills:   0%|                                                           | 2/9646 [00:17<22:47:50,  8.51s/it]

Error parsing job 2 (skipped): Invalid json output: 
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 


Extracting skills:   0%|                                                           | 3/9646 [00:25<22:23:51,  8.36s/it]

...

Error parsing job 8095 (skipped): Invalid json output: 
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 


Extracting skills:  84%|█████████████████████████████████████████████▎        | 8096/9646 [19:08:55<3:42:49,  8.63s/it]

Currently jobs added: 4902


Extracting skills:  84%|█████████████████████████████████████████████▎        | 8096/9646 [19:09:02<3:39:59,  8.52s/it]


KeyboardInterrupt: 